In [1]:
import torch
from torchinfo import summary
dino_model_dict = {
    "vits": 'dinov2_vits14',
    "vitb": 'dinov2_vitb14',
    "vitl": 'dinov2_vitl14',
    "vitg": 'dinov2_vitg14',
    "vits_reg": 'dinov2_vits14_reg',
    "vitb_reg": 'dinov2_vitb14_reg',
    "vitl_reg": 'dinov2_vitl14_reg',
    "vitg_reg": 'dinov2_vitg14_reg',
}




In [5]:
def load_dino_model(model_name, pretrained=True):

    model = torch.hub.load('facebookresearch/dinov2', dino_model_dict[model_name], pretrained=pretrained)
    return model

model = load_dino_model("vits")
model(torch.randn(1, 3, 252, 252))
summary(model, input_size=(1, 3, 252, 252))

KeyboardInterrupt: 

我们看一下 Dino 

In [ ]:
import torch
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.image as mpimg 
from PIL import Image
from sklearn.decomposition import PCA
import matplotlib
torch.cuda.set_device(4)
# 设置补丁(patch)的高度和宽度
patch_h = 60
patch_w = 80
# 特征维度
feat_dim = 384

# 定义图像转换操作
transform = T.Compose([
    T.GaussianBlur(9, sigma=(0.1, 2.0)),  # 高斯模糊
    T.Resize((patch_h * 14, patch_w * 14)),  # 调整图像大小
    T.CenterCrop((patch_h * 14, patch_w * 14)),  # 中心裁剪
    T.ToTensor(),  # 转换为张量
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),  # 标准化
])

# 使用torch.hub加载dinov2_vits14模型并移至CUDA设备
dinov2_vits14 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14', pretrained=True).cuda()

In [ ]:
# 创建用于存储特征和图像张量的零张量
features = torch.zeros(4, patch_h * patch_w, feat_dim)
print(f"Feature has shape: {features.shape}")
imgs_tensor = torch.zeros(4, 3, patch_h * 14, patch_w * 14).cuda()
print(f"Image tensor has shape: {imgs_tensor.shape}")
# 图像路径
# img_path = f'data/dream/real/panda-3cam_azure/000000.rgb.jpg'
img_path = "data/dream/synthetic/panda_synth_test_dr/000001.rgb.jpg"
# 打开图像并转换为RGB模式
img = Image.open(img_path).convert('RGB')
print(img.size)
# 对图像进行转换操作，并将其存储在imgs_tensor的第一个位置
imgs_tensor[0] = transform(img)[:3]

# 禁用梯度计算
with torch.no_grad():
    # 将图像张量传递给dinov2_vits14模型获取特征
    features_dict = dinov2_vits14.forward_features(imgs_tensor)
    features = features_dict['x_norm_patchtokens']
    
print(f"features output is a dict with keys: {features_dict.keys()}")
print(f"features norm patchtoken shape: {features.shape}")
# 重塑特征形状为(4 * patch_h * patch_w, feat_dim)
features = features.reshape(4 * patch_h * patch_w, feat_dim).cpu()
print(f"Feature has shape: {features.shape}")
# 创建PCA对象并拟合特征
pca = PCA(n_components=3)
pca.fit(features)

# 对PCA转换后的特征进行归一化处理
pca_features = pca.transform(features)
pca_features[:, 0] = (pca_features[:, 0] - pca_features[:, 0].min()) / (pca_features[:, 0].max() - pca_features[:, 0].min())

# 根据阈值进行前景和背景的区分
pca_features_fg = pca_features[:, 0] > 0.3
pca_features_bg = ~pca_features_fg

# 查找背景特征的索引
b = np.where(pca_features_bg)

# 对前景特征再次进行PCA转换
pca.fit(features[pca_features_fg])
pca_features_rem = pca.transform(features[pca_features_fg])

# 对前景特征进行归一化处理
for i in range(3):
    pca_features_rem[:, i] = (pca_features_rem[:, i] - pca_features_rem[:, i].min()) / (pca_features_rem[:, i].max() - pca_features_rem[:, i].min())
    # 使用均值和标准差进行转换，个人发现这种转换方式可以得到更好的可视化效果
    # pca_features_rem[:, i] = (pca_features_rem[:, i] - pca_features_rem[:, i].mean()) / (pca_features_rem[:, i].std() ** 2) + 0.5

# 创建RGB特征数组
pca_features_rgb = pca_features.copy()

# 替换前景特征为转换后的特征
pca_features_rgb[pca_features_fg] = pca_features_rem

# 将背景特征设置为0
pca_features_rgb[b] = 0

# 重塑特征形状为(4, patch_h, patch_w, 3)
pca_features_rgb = pca_features_rgb.reshape(4, patch_h, patch_w, 3)

# 显示第一个图像的RGB特征
plt.imshow(pca_features_rgb[0][...,::-1])
plt.savefig('features.png')
plt.show()
plt.close()


NameError: name 'torch' is not defined